# Random Forest для прогноза золото-урановых зон — исправленная версия

Ноутбук строит сетку, считает признаки расстояний до геологических факторов, обучает **Random Forest** по точкам проявлений/аномалий и сохраняет карту прогноза.

Что исправлено:
- защита от ошибки `ValueError: arange: cannot compute length`;
- проверка пустых слоёв и битых геометрий;
- автоматическое приведение CRS;
- более безопасное построение сетки;
- сохранение PNG, GPKG, CSV и JSON с метриками.

In [1]:
# Если библиотек нет, раскомментируй и выполни:
# !pip install geopandas shapely scikit-learn matplotlib pandas numpy pyogrio

import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.ops import unary_union
from shapely.errors import GEOSException

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

## 1. Настройки

Проверь только `BASE_DIR`. Внутри должна быть папка `shp_dbf` с shapefile.

In [2]:
# === ГЛАВНАЯ НАСТРОЙКА ===
BASE_DIR = r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"
SHP_DIR = os.path.join(BASE_DIR, "shp_dbf")
OUT_DIR = os.path.join(BASE_DIR, "rf_forecast_result_fixed")
os.makedirs(OUT_DIR, exist_ok=True)

# Размер ячейки в метрах
CELL_SIZE = 500

# Минимальное число positive-ячеек, чтобы обучать ML
MIN_POSITIVES_FOR_ML = 5

# Random Forest
RF_PARAMS = dict(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUT_DIR:", OUT_DIR)

BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
SHP_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\shp_dbf
OUT_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_forecast_result_fixed


## 2. Вспомогательные функции

In [3]:
def normalize_name(s):
    return str(s).lower().replace("ё", "е")


def find_shp_files(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Папка не найдена: {folder}")
    files = sorted(folder.glob("*.shp"))
    if not files:
        raise FileNotFoundError(f"В папке нет .shp файлов: {folder}")
    return files


def repair_geometries(gdf):
    """Чистка геометрий: удаляем пустые, исправляем buffer(0)."""
    if gdf is None or gdf.empty:
        return gdf
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    if gdf.empty:
        return gdf
    try:
        invalid = ~gdf.geometry.is_valid
        if invalid.any():
            gdf.loc[invalid, "geometry"] = gdf.loc[invalid, "geometry"].buffer(0)
    except Exception:
        pass
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    return gdf.reset_index(drop=True)


def load_layer(path, fallback_crs="EPSG:4326"):
    """Безопасно загружает слой."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")
    gdf = gpd.read_file(path)
    gdf = repair_geometries(gdf)
    if gdf is None or gdf.empty:
        print(f"⚠️ Слой пустой после загрузки: {path.name}")
        return gdf
    if gdf.crs is None:
        print(f"⚠️ CRS не найден → задаю {fallback_crs}: {path.name}")
        gdf = gdf.set_crs(fallback_crs, allow_override=True)
    return gdf


def choose_projected_crs(mask_gdf):
    """Выбирает метрическую CRS. Если исходная CRS географическая, пытается подобрать UTM."""
    if mask_gdf.crs is None:
        mask_gdf = mask_gdf.set_crs("EPSG:4326", allow_override=True)
    try:
        if mask_gdf.crs.is_projected:
            return mask_gdf.crs
    except Exception:
        pass
    try:
        estimated = mask_gdf.estimate_utm_crs()
        if estimated is not None:
            return estimated
    except Exception:
        pass
    return "EPSG:3857"


def to_crs_safe(gdf, crs, name="layer"):
    if gdf is None or gdf.empty:
        return gdf
    if gdf.crs is None:
        print(f"⚠️ CRS отсутствует у {name}, задаю EPSG:4326")
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    try:
        return repair_geometries(gdf.to_crs(crs))
    except Exception as e:
        raise RuntimeError(f"Не удалось перепроецировать {name}: {e}")


def finite_bounds(gdf, layer_name="mask"):
    if gdf is None or gdf.empty:
        raise ValueError(f"Слой {layer_name} пустой — границы построить нельзя")
    bounds = np.asarray(gdf.total_bounds, dtype=float)
    if len(bounds) != 4 or not np.all(np.isfinite(bounds)):
        raise ValueError(f"Некорректные total_bounds у {layer_name}: {bounds}")
    minx, miny, maxx, maxy = bounds
    if minx >= maxx or miny >= maxy:
        raise ValueError(f"Некорректные границы у {layer_name}: {bounds}")
    return minx, miny, maxx, maxy


def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    mn, mx = np.min(arr), np.max(arr)
    if mx - mn < 1e-12:
        return np.zeros_like(arr, dtype=float)
    return (arr - mn) / (mx - mn)


def distance_to_union(geoms, target_gdf):
    """Расстояние от каждой геометрии/точки до объединения target_gdf."""
    if target_gdf is None or target_gdf.empty:
        return np.full(len(geoms), np.nan)
    target_gdf = repair_geometries(target_gdf)
    if target_gdf.empty:
        return np.full(len(geoms), np.nan)
    try:
        union = target_gdf.geometry.unary_union
    except Exception:
        union = unary_union(list(target_gdf.geometry))
    return np.array([geom.distance(union) for geom in geoms], dtype=float)


def proximity_from_distance(dist, scale):
    dist = np.asarray(dist, dtype=float)
    dist = np.nan_to_num(dist, nan=np.nanmax(dist[np.isfinite(dist)]) if np.isfinite(dist).any() else 1e9)
    return 1.0 / (1.0 + dist / float(scale))

## 3. Загрузка слоёв

Если названия файлов отличаются, измени словарь `LAYER_FILES`.

In [4]:
all_shp = find_shp_files(SHP_DIR)
print("Найденные shp-файлы:")
for p in all_shp:
    print(" -", p.name)

# Основные слои факторов. При необходимости поменяй имена файлов.
LAYER_FILES = {
    "mask": "svita_new.shp",          # область прогноза / свиты
    "facies": "fasii.shp",           # литолого-фациальный фактор
    "paleo": "gr_dol_vp_poly.shp",   # палеодолины / впадины
    "struct": "kory.shp",            # коры выветривания
    "magm": "dayki_buf.shp",         # дайки
    "tect1": "glub_raz_nw.shp",      # глубинные разломы
    "tect2": "glub_r_nw.shp",        # региональные разломы
}

layers = {}
for key, filename in LAYER_FILES.items():
    path = Path(SHP_DIR) / filename
    if path.exists():
        layers[key] = load_layer(path)
        print(f"✅ {key:7s}: {filename:25s} | объектов: {len(layers[key])}")
    else:
        layers[key] = None
        print(f"⚠️ {key:7s}: файл не найден: {filename}")

if layers["mask"] is None or layers["mask"].empty:
    raise ValueError("Не найден или пустой mask-слой. Проверь LAYER_FILES['mask'].")

Найденные shp-файлы:
 - dayki_buf.shp
 - fasii.shp
 - glub_r_nw.shp
 - glub_raz_nw.shp
 - gr_dol_vp_poly.shp
 - kory.shp
 - result.shp
 - svita_new.shp
 - геохимические ореолы.shp
 - геохимическое_опробование.shp
 - привнос урана.shp
⚠️ CRS не найден → задаю EPSG:4326: svita_new.shp
✅ mask   : svita_new.shp             | объектов: 101
⚠️ CRS не найден → задаю EPSG:4326: fasii.shp
✅ facies : fasii.shp                 | объектов: 15
⚠️ CRS не найден → задаю EPSG:4326: gr_dol_vp_poly.shp
✅ paleo  : gr_dol_vp_poly.shp        | объектов: 6
⚠️ CRS не найден → задаю EPSG:4326: kory.shp
✅ struct : kory.shp                  | объектов: 19
⚠️ CRS не найден → задаю EPSG:4326: dayki_buf.shp
✅ magm   : dayki_buf.shp             | объектов: 151
⚠️ CRS не найден → задаю EPSG:4326: glub_raz_nw.shp
✅ tect1  : glub_raz_nw.shp           | объектов: 4
⚠️ CRS не найден → задаю EPSG:4326: glub_r_nw.shp
✅ tect2  : glub_r_nw.shp             | объектов: 2


## 4. Приведение CRS к метрам

In [5]:
TARGET_CRS = choose_projected_crs(layers["mask"])
print("Выбранная метрическая CRS:", TARGET_CRS)

for key in list(layers.keys()):
    if layers[key] is not None and not layers[key].empty:
        layers[key] = to_crs_safe(layers[key], TARGET_CRS, key)
        print(f"{key:7s} CRS → {layers[key].crs}")

print("Mask bounds:", layers["mask"].total_bounds)

Выбранная метрическая CRS: EPSG:3857
mask    CRS → EPSG:3857
facies  CRS → EPSG:3857
paleo   CRS → EPSG:3857
struct  CRS → EPSG:3857
magm    CRS → EPSG:3857
tect1   CRS → EPSG:3857
tect2   CRS → EPSG:3857
Mask bounds: [inf inf inf inf]


## 5. Построение сетки 500×500 м

Здесь исправлена ошибка `np.arange: cannot compute length`: перед построением сетки проверяются границы, CRS и размер области.

In [6]:
def build_grid(mask_gdf, cell_size=500):
    mask_gdf = repair_geometries(mask_gdf)
    minx, miny, maxx, maxy = finite_bounds(mask_gdf, "mask")

    width = maxx - minx
    height = maxy - miny
    print(f"Размер области: width={width:.2f} м, height={height:.2f} м")

    if not np.isfinite(cell_size) or cell_size <= 0:
        raise ValueError(f"Некорректный CELL_SIZE: {cell_size}")

    if width < cell_size or height < cell_size:
        old = cell_size
        cell_size = max(width, height) / 50
        print(f"⚠️ Область меньше исходной ячейки. CELL_SIZE изменён: {old} → {cell_size:.2f}")

    # Безопасное число колонок и строк вместо np.arange без проверки.
    n_cols = int(math.ceil(width / cell_size))
    n_rows = int(math.ceil(height / cell_size))

    if n_cols <= 0 or n_rows <= 0:
        raise ValueError(f"Сетка не строится: n_cols={n_cols}, n_rows={n_rows}")

    if n_cols * n_rows > 2_000_000:
        raise ValueError(
            f"Слишком много ячеек: {n_cols*n_rows}. "
            f"Увеличь CELL_SIZE или проверь CRS/границы."
        )

    try:
        mask_union = mask_gdf.geometry.unary_union
    except Exception:
        mask_union = unary_union(list(mask_gdf.geometry))

    records = []
    idx = 0
    for row in range(n_rows):
        y1 = miny + row * cell_size
        y2 = min(y1 + cell_size, maxy)
        for col in range(n_cols):
            x1 = minx + col * cell_size
            x2 = min(x1 + cell_size, maxx)
            geom = box(x1, y1, x2, y2)
            # Оставляем только ячейки, которые пересекают маску
            if geom.intersects(mask_union):
                records.append({
                    "cell_id": idx,
                    "row": row,
                    "col": col,
                    "geometry": geom.intersection(mask_union),
                })
                idx += 1

    grid = gpd.GeoDataFrame(records, crs=mask_gdf.crs)
    grid = repair_geometries(grid)
    if grid.empty:
        raise ValueError("После пересечения с маской сетка пустая. Проверь CRS и mask.")
    return grid


grid = build_grid(layers["mask"], CELL_SIZE)
print("Ячеек в сетке:", len(grid))
grid.head()

ValueError: Некорректные total_bounds у mask: [inf inf inf inf]

## 6. Расчёт признаков по расстояниям

In [ ]:
# Масштабы расстояний в метрах. Их можно менять.
SCALES = {
    "facies": 1000,
    "paleo": 1000,
    "struct": 900,
    "magm": 800,
    "tect1": 800,
    "tect2": 800,
}

centroids = grid.geometry.centroid

for key, scale in SCALES.items():
    layer = layers.get(key)
    dist_col = f"dist_{key}"
    prox_col = f"prox_{key}"

    if layer is None or layer.empty:
        print(f"⚠️ {key}: слой отсутствует → признак будет 0")
        grid[dist_col] = np.nan
        grid[prox_col] = 0.0
        continue

    dist = distance_to_union(centroids, layer)
    grid[dist_col] = dist
    grid[prox_col] = proximity_from_distance(dist, scale)
    print(f"✅ {key}: {prox_col} рассчитан")

# Взаимодействия факторов
for col in ["prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect1", "prox_tect2"]:
    if col not in grid.columns:
        grid[col] = 0.0

grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = ((grid["prox_tect1"] + grid["prox_tect2"]) / 2) * grid["prox_magm"]
grid["tect_struct_intersection"] = ((grid["prox_tect1"] + grid["prox_tect2"]) / 2) * grid["prox_struct"]
grid["paleo_struct_intersection"] = grid["prox_paleo"] * grid["prox_struct"]

FEATURES = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect1", "prox_tect2",
    "tect_intersection", "tect_magm_intersection", "tect_struct_intersection", "paleo_struct_intersection"
]

grid[FEATURES].describe()

## 7. Поиск точечных слоёв с проявлениями / аномалиями

Ноутбук автоматически ищет point-слои. Если нужный слой не найден, можно вручную указать его имя в `EVIDENCE_LAYER_NAMES`.

In [ ]:
# Можно вручную указать конкретные точечные слои, например:
# EVIDENCE_LAYER_NAMES = ["result.shp", "layer_01.shp"]
EVIDENCE_LAYER_NAMES = []

KEYWORDS = ["au", "gold", "zolot", "зол", "uran", "uranium", "уран", "u_"]

def looks_like_evidence_file(path):
    name = normalize_name(path.stem)
    return any(k in name for k in KEYWORDS) or name in ["result", "layer_00", "layer_01", "layer_02"]

point_layers = []
for path in all_shp:
    if EVIDENCE_LAYER_NAMES and path.name not in EVIDENCE_LAYER_NAMES:
        continue
    try:
        gdf = load_layer(path)
        if gdf is None or gdf.empty:
            continue
        geom_types = set(gdf.geometry.geom_type.astype(str).str.lower())
        is_point = any("point" in t for t in geom_types)
        if is_point and (EVIDENCE_LAYER_NAMES or looks_like_evidence_file(path)):
            point_layers.append((path.name, gdf))
            print(f"✅ кандидат evidence: {path.name}, объектов: {len(gdf)}, geom_types: {geom_types}")
    except Exception as e:
        print(f"⚠️ Не удалось проверить {path.name}: {e}")

if not point_layers:
    print("⚠️ Точечные evidence-слои автоматически не найдены.")
    print("   Укажи вручную EVIDENCE_LAYER_NAMES = ['имя_слоя.shp'] и перезапусти ячейку.")
else:
    print("Всего evidence-слоёв:", len(point_layers))

## 8. Создание target для ML

In [ ]:
def combine_evidence_layers(point_layers, target_crs):
    frames = []
    for name, gdf in point_layers:
        gdf = to_crs_safe(gdf, target_crs, name)
        gdf = repair_geometries(gdf)
        if gdf is None or gdf.empty:
            continue
        # Оставляем только точки
        gdf = gdf[gdf.geometry.geom_type.str.contains("Point", case=False, na=False)].copy()
        if gdf.empty:
            continue
        gdf["source_layer"] = name
        frames.append(gdf[["source_layer", "geometry"]])
    if not frames:
        return gpd.GeoDataFrame(columns=["source_layer", "geometry"], geometry="geometry", crs=target_crs)
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=target_crs)


evidence = combine_evidence_layers(point_layers, grid.crs)
print("Evidence points:", len(evidence))

# Spatial join: есть ли точка внутри ячейки
if len(evidence) > 0:
    join = gpd.sjoin(
        grid[["cell_id", "geometry"]],
        evidence[["source_layer", "geometry"]],
        how="left",
        predicate="contains",
    )
    positive_ids = set(join.loc[join["index_right"].notna(), "cell_id"].astype(int))
    grid["target"] = grid["cell_id"].isin(positive_ids).astype(int)
else:
    grid["target"] = 0

n_pos = int(grid["target"].sum())
n_total = len(grid)
print("Positive cells:", n_pos)
print("Total cells:", n_total)
print("Base rate:", n_pos / n_total if n_total else 0)

# Проверка визуально: точки + сетка
fig, ax = plt.subplots(figsize=(9, 9))
grid.boundary.plot(ax=ax, linewidth=0.05)
if len(evidence) > 0:
    evidence.plot(ax=ax, markersize=8)
ax.set_title("Сетка и evidence-точки")
ax.set_axis_off()
plt.show()

## 9. Экспертный fallback-score

Если положительных точек слишком мало, ML не сможет нормально обучиться. Тогда карта всё равно строится по геологическому скору, а не падает.

In [ ]:
geo_raw = (
    1.4 * grid["prox_tect1"] +
    1.4 * grid["prox_tect2"] +
    1.2 * grid["prox_magm"] +
    1.1 * grid["prox_struct"] +
    1.0 * grid["prox_paleo"] +
    0.9 * grid["prox_facies"] +
    0.5 * grid["tect_intersection"] +
    0.4 * grid["tect_magm_intersection"] +
    0.3 * grid["tect_struct_intersection"] +
    0.2 * grid["paleo_struct_intersection"]
)

grid["geo_score"] = normalize_01(geo_raw)
grid[["geo_score"]].describe()

## 10. Обучение Random Forest

In [ ]:
X = grid[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)
y = grid["target"].astype(int)

use_ml = (y.sum() >= MIN_POSITIVES_FOR_ML) and (y.nunique() == 2)
metrics = {
    "n_cells": int(len(grid)),
    "n_positive_cells": int(y.sum()),
    "base_rate": float(y.mean()) if len(y) else 0.0,
    "use_ml": bool(use_ml),
    "features": FEATURES,
    "rf_params": RF_PARAMS,
}

if use_ml:
    print("✅ Данных достаточно — обучаю Random Forest")

    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X, y)

    grid["rf_score"] = rf.predict_proba(X)[:, 1]
    grid["prospectivity"] = grid["rf_score"]
    grid["prognoz"] = 1 - grid["prospectivity"]  # для совместимости с методичкой: меньше = перспективнее

    # Оценка на holdout, если возможно
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        rf_test = RandomForestClassifier(**RF_PARAMS)
        rf_test.fit(X_train, y_train)
        proba_test = rf_test.predict_proba(X_test)[:, 1]
        pred_test = (proba_test >= 0.5).astype(int)

        metrics["ROC_AUC_holdout"] = float(roc_auc_score(y_test, proba_test))
        metrics["Average_Precision_holdout"] = float(average_precision_score(y_test, proba_test))
        metrics["Precision_05_holdout"] = float(precision_score(y_test, pred_test, zero_division=0))
        metrics["Recall_05_holdout"] = float(recall_score(y_test, pred_test, zero_division=0))
        metrics["F1_05_holdout"] = float(f1_score(y_test, pred_test, zero_division=0))
        metrics["ConfusionMatrix_holdout"] = confusion_matrix(y_test, pred_test).tolist()
    except Exception as e:
        metrics["holdout_error"] = str(e)
        print("⚠️ Holdout-метрики не рассчитаны:", e)

    # Feature importance
    importance = pd.DataFrame({
        "feature": FEATURES,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)
    display(importance)

else:
    print("⚠️ Недостаточно positive-точек для ML. Использую geo_score как fallback.")
    grid["rf_score"] = np.nan
    grid["prospectivity"] = grid["geo_score"]
    grid["prognoz"] = 1 - grid["prospectivity"]
    importance = pd.DataFrame({"feature": FEATURES, "importance": np.nan})

metrics

## 11. Top-10% перспективных зон

In [ ]:
threshold_top10 = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= threshold_top10).astype(int)

if grid["target"].sum() > 0:
    hit_rate_top10 = float(grid.loc[grid["target"] == 1, "top10"].mean())
    lift_top10 = hit_rate_top10 / 0.10
else:
    hit_rate_top10 = None
    lift_top10 = None

metrics["Top10_threshold"] = threshold_top10
metrics["HitRate_Top10"] = hit_rate_top10
metrics["Lift_Top10"] = lift_top10

print("Top10 threshold:", threshold_top10)
print("HitRate Top10:", hit_rate_top10)
print("Lift Top10:", lift_top10)

## 12. Визуализация карты прогноза

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

grid.plot(
    column="prospectivity",
    ax=ax,
    cmap="YlOrRd",
    legend=True,
    linewidth=0,
    legend_kwds={"label": "Prospectivity: 0 — низкая, 1 — высокая"},
)

# Граница области
layers["mask"].boundary.plot(ax=ax, linewidth=0.6)

# Evidence-точки
if len(evidence) > 0:
    evidence.plot(ax=ax, markersize=18, edgecolor="black", linewidth=0.4)

ax.set_title("Random Forest prospectivity map")
ax.set_axis_off()
plt.tight_layout()

png_path = os.path.join(OUT_DIR, "rf_prospectivity_map.png")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print("PNG сохранён:", png_path)

## 13. Визуализация Top-10% зон

In [ ]:
top10 = grid[grid["top10"] == 1].copy()

fig, ax = plt.subplots(figsize=(12, 10))
grid.plot(ax=ax, color="lightgray", linewidth=0)
top10.plot(ax=ax, color="gold", edgecolor="black", linewidth=0.1)
layers["mask"].boundary.plot(ax=ax, linewidth=0.6)

if len(evidence) > 0:
    evidence.plot(ax=ax, markersize=18, edgecolor="black", linewidth=0.4)

ax.set_title("Top-10% перспективных зон")
ax.set_axis_off()
plt.tight_layout()

top_png_path = os.path.join(OUT_DIR, "rf_top10_zones.png")
plt.savefig(top_png_path, dpi=300, bbox_inches="tight")
plt.show()

print("PNG сохранён:", top_png_path)

## 14. Сохранение результатов

In [ ]:
# Сохраняем только нужные колонки + геометрию
save_cols = [
    "cell_id", "row", "col", "target", "geo_score", "rf_score", "prospectivity", "prognoz", "top10"
] + FEATURES + ["geometry"]

result_gpkg = os.path.join(OUT_DIR, "rf_forecast_result.gpkg")
result_csv = os.path.join(OUT_DIR, "rf_grid_attributes.csv")
metrics_json = os.path.join(OUT_DIR, "rf_metrics.json")
importance_csv = os.path.join(OUT_DIR, "rf_feature_importance.csv")

# GPKG
grid[save_cols].to_file(result_gpkg, layer="rf_grid", driver="GPKG")
if len(evidence) > 0:
    evidence.to_file(result_gpkg, layer="evidence_points", driver="GPKG")
top10[save_cols].to_file(result_gpkg, layer="top10_zones", driver="GPKG")

# CSV без геометрии
grid.drop(columns="geometry").to_csv(result_csv, index=False, encoding="utf-8-sig")
importance.to_csv(importance_csv, index=False, encoding="utf-8-sig")

with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Готово!")
print("GPKG:", result_gpkg)
print("CSV:", result_csv)
print("Metrics:", metrics_json)
print("Importance:", importance_csv)
print("PNG map:", png_path)
print("PNG top10:", top_png_path)

## 15. Короткая интерпретация для презентации

In [ ]:
print("ИТОГ ДЛЯ ПРЕЗЕНТАЦИИ")
print("Метод: Random Forest")
print("Ячеек:", metrics["n_cells"])
print("Positive-ячеек:", metrics["n_positive_cells"])
print("ML использован:", metrics["use_ml"])

for k, v in metrics.items():
    if "AUC" in k or "Precision" in k or "Recall" in k or "F1" in k or "Top10" in k or "Lift" in k:
        print(f"{k}: {v}")

if metrics["use_ml"]:
    print("\nВывод: Random Forest выбран, потому что умеет учитывать нелинейные связи между геологическими факторами.")
else:
    print("\nВывод: ML не был обучен из-за малого числа positive-точек; использован геологический скор как fallback.")